# Chapter 18: Reinforcement Learning - Part 1

## Introduction to Reinforcement Learning

Reinforcement Learning (RL) is one of the most exciting fields in Machine Learning. The breakthrough came in 2013 when DeepMind demonstrated a system that could learn to play Atari games from scratch using only raw pixels, eventually outperforming humans. This led to AlphaGo's victory against world champions in 2016-2017.

## Learning to Optimize Rewards

In Reinforcement Learning, a software agent:
- Makes **observations** within an environment
- Takes **actions** based on those observations
- Receives **rewards** (positive or negative)
- Learns to maximize expected rewards over time

### Key Applications
- **Robotics**: Agents control robots using sensors and motors, receiving rewards for reaching goals
- **Game Playing**: Ms. Pac-Man, Go, Atari games
- **Control Systems**: Smart thermostats anticipating user needs
- **Financial Trading**: Stock market decisions with monetary gains/losses
- **Self-driving cars, recommender systems, ad placement**

## Policy Search

The **policy** is the algorithm an agent uses to determine actions. It can be:
- **Deterministic**: Always chooses the same action for a given state
- **Stochastic**: Involves randomness in action selection

### Policy Search Approaches

1. **Brute Force**: Try many parameter combinations, pick the best
2. **Genetic Algorithms**: Create generations of policies, keep the best performers, create offspring with variations
3. **Policy Gradients (PG)**: Evaluate gradients of rewards with respect to policy parameters, follow gradients toward higher rewards

## OpenAI Gym Setup

OpenAI Gym provides simulated environments for training RL agents. Let's install the necessary packages and set up our environment.

In [ ]:
# Install required packages
!pip install gym pygame numpy matplotlib

In [ ]:
# Import necessary libraries
import gym
import numpy as np
import matplotlib.pyplot as plt
from IPython import display

print("Libraries imported successfully!")
print(f"Gym version: {gym.__version__}")

# Detect Gym API version
try:
    gym_version = tuple(map(int, gym.__version__.split('.')[:2]))
    NEW_GYM_API = gym_version >= (0, 26)
except:
    NEW_GYM_API = False

if NEW_GYM_API:
    print("\n📝 Using NEW Gym API (v0.26+)")
    print("   - env.reset() returns (observation, info)")
    print("   - env.step() returns (obs, reward, terminated, truncated, info)")
else:
    print("\n📝 Using OLD Gym API (< v0.26)")
    print("   - env.reset() returns observation")
    print("   - env.step() returns (obs, reward, done, info)")

# Helper functions for API compatibility
def reset_env(env):
    """Reset environment, compatible with both old and new Gym API."""
    result = env.reset()
    if isinstance(result, tuple):
        return result[0], result[1]  # New API: (obs, info)
    else:
        return result, {}  # Old API: obs

def step_env(env, action):
    """Step environment, compatible with both old and new Gym API."""
    result = env.step(action)
    if len(result) == 5:
        obs, reward, terminated, truncated, info = result
        return obs, reward, terminated or truncated, info  # New API
    else:
        return result  # Old API: (obs, reward, done, info)

print("\n✅ Compatibility helpers loaded!")

Libraries imported successfully!
Gym version: 0.26.2

📝 Using NEW Gym API (v0.26+)
   - env.reset() returns (observation, info)
   - env.step() returns (obs, reward, terminated, truncated, info)


## Creating and Exploring an Environment

Let's create our first environment - CartPole. In this environment:
- A pole is attached to a cart that moves along a frictionless track
- The agent must keep the pole balanced by moving the cart left or right
- The episode ends if the pole falls too far or the cart moves off screen

In [ ]:
# Create the CartPole environment
env = gym.make("CartPole-v1")

# Reset the environment to get initial observation
reset_result = env.reset()
if isinstance(reset_result, tuple):
    obs, info = reset_result  # New API
else:
    obs, info = reset_result, {}  # Old API

print("Initial observation:", obs)
print("\nObservation shape:", obs.shape)
print("\nObservation components:")
print("  [0] Cart position:", obs[0])
print("  [1] Cart velocity:", obs[1])
print("  [2] Pole angle:", obs[2])
print("  [3] Pole angular velocity:", obs[3])

Initial observation: [ 0.02932168 -0.040914   -0.03466681  0.03845349]

Observation shape: (4,)

Observation components:
  [0] Cart position: 0.02932168
  [1] Cart velocity: -0.040914
  [2] Pole angle: -0.034666806
  [3] Pole angular velocity: 0.03845349


### Understanding the Action Space

The CartPole environment has a discrete action space with 2 possible actions:
- **0**: Push cart to the left
- **1**: Push cart to the right

In [8]:
# Check the action space
print("Action space:", env.action_space)
print("Number of actions:", env.action_space.n)
print("\nActions:")
print("  0 = Push left")
print("  1 = Push right")

Action space: Discrete(2)
Number of actions: 2

Actions:
  0 = Push left
  1 = Push right


### Taking Actions in the Environment

When we take an action, the environment returns:
- **observation**: New state of the environment
- **reward**: Reward received for the action (1.0 for each step the pole stays upright)
- **done**: Boolean indicating if the episode is finished
- **info**: Additional diagnostic information (not used in CartPole)

In [13]:
# Reset environment
reset_result = env.reset()
if isinstance(reset_result, tuple):
    obs, info = reset_result
else:
    obs, info = reset_result, {}
print("Initial state:", obs)

# Take action 1 (push right)
action = 1
step_result = env.step(action)
if len(step_result) == 5:
    obs, reward, terminated, truncated, info = step_result
    done = terminated or truncated
else:
    obs, reward, done, info = step_result

print("\nAfter taking action 1 (push right):")
print("New observation:", obs)
print("Reward received:", reward)
print("Episode done:", done)
print("Info:", info)

Initial state: [-0.0292567  -0.02691031 -0.03250463  0.00088477]

After taking action 1 (push right):
New observation: [-0.0297949   0.16866237 -0.03248693 -0.30187395]
Reward received: 1.0
Episode done: False
Info: {}


## Simple Hard-Coded Policy

Let's implement a simple policy that uses only the pole angle to make decisions:
- If the pole is leaning left (angle < 0), push left
- If the pole is leaning right (angle > 0), push right

This is a deterministic policy based on a single observation feature.

In [14]:
def basic_policy(obs):
    """
    Simple policy based on pole angle.
    
    Args:
        obs: Observation array [cart_pos, cart_vel, pole_angle, pole_vel]
    
    Returns:
        action: 0 (left) if angle < 0, else 1 (right)
    """
    angle = obs[2]  # pole angle is at index 2
    return 0 if angle < 0 else 1

# Test the policy
test_obs = np.array([0.0, 0.0, -0.05, 0.0])  # pole leaning left
print(f"Observation: {test_obs}")
print(f"Pole angle: {test_obs[2]}")
print(f"Action chosen: {basic_policy(test_obs)} (0=left, 1=right)")

test_obs = np.array([0.0, 0.0, 0.05, 0.0])  # pole leaning right
print(f"\nObservation: {test_obs}")
print(f"Pole angle: {test_obs[2]}")
print(f"Action chosen: {basic_policy(test_obs)} (0=left, 1=right)")

Observation: [ 0.    0.   -0.05  0.  ]
Pole angle: -0.05
Action chosen: 0 (0=left, 1=right)

Observation: [0.   0.   0.05 0.  ]
Pole angle: 0.05
Action chosen: 1 (0=left, 1=right)


### Evaluating the Basic Policy

Let's run multiple episodes with our basic policy and see how well it performs. We'll measure:
- **Episode reward**: Total reward accumulated in an episode
- **Episode length**: Number of steps before the pole falls

The goal in CartPole-v1 is to keep the pole balanced for 200 steps (maximum reward = 200).

In [ ]:
# Evaluate the basic policy over multiple episodes
n_episodes = 500
max_steps = 200
totals = []

for episode in range(n_episodes):
    episode_rewards = 0
    obs, info = reset_env(env)
    
    for step in range(max_steps):
        action = basic_policy(obs)
        obs, reward, done, info = step_env(env, action)
        episode_rewards += reward
        
        if done:
            break
    
    totals.append(episode_rewards)

# Calculate statistics
print(f"Evaluated {n_episodes} episodes")
print(f"Mean reward: {np.mean(totals):.2f}")
print(f"Std reward: {np.std(totals):.2f}")
print(f"Min reward: {np.min(totals):.2f}")
print(f"Max reward: {np.max(totals):.2f}")

ValueError: too many values to unpack (expected 4)

In [ ]:
# Visualize the performance
plt.figure(figsize=(12, 5))

# Plot 1: Rewards over episodes
plt.subplot(1, 2, 1)
plt.plot(totals, alpha=0.6)
plt.axhline(y=np.mean(totals), color='r', linestyle='--', label=f'Mean: {np.mean(totals):.2f}')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Basic Policy Performance Over Episodes')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Distribution of rewards
plt.subplot(1, 2, 2)
plt.hist(totals, bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Total Reward')
plt.ylabel('Frequency')
plt.title('Distribution of Episode Rewards')
plt.axvline(x=np.mean(totals), color='r', linestyle='--', label=f'Mean: {np.mean(totals):.2f}')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Visualizing a Single Episode

Let's visualize what our basic policy does in a single episode, showing the state trajectory.

In [ ]:
# Run one episode and collect state information
obs, info = env.reset()
states = []
actions_taken = []
rewards_received = []

for step in range(200):
    action = basic_policy(obs)
    states.append(obs.copy())
    actions_taken.append(action)
    
    obs, reward, done, info = env.step(action)
    rewards_received.append(reward)
    
    if done:
        break

states = np.array(states)
print(f"Episode lasted {len(states)} steps")
print(f"Total reward: {sum(rewards_received)}")

In [ ]:
# Visualize the state trajectory
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Cart position
axes[0, 0].plot(states[:, 0])
axes[0, 0].set_xlabel('Step')
axes[0, 0].set_ylabel('Cart Position')
axes[0, 0].set_title('Cart Position Over Time')
axes[0, 0].grid(True, alpha=0.3)

# Cart velocity
axes[0, 1].plot(states[:, 1], color='orange')
axes[0, 1].set_xlabel('Step')
axes[0, 1].set_ylabel('Cart Velocity')
axes[0, 1].set_title('Cart Velocity Over Time')
axes[0, 1].grid(True, alpha=0.3)

# Pole angle
axes[1, 0].plot(states[:, 2], color='green')
axes[1, 0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[1, 0].set_xlabel('Step')
axes[1, 0].set_ylabel('Pole Angle (radians)')
axes[1, 0].set_title('Pole Angle Over Time')
axes[1, 0].grid(True, alpha=0.3)

# Pole angular velocity
axes[1, 1].plot(states[:, 3], color='red')
axes[1, 1].set_xlabel('Step')
axes[1, 1].set_ylabel('Pole Angular Velocity')
axes[1, 1].set_title('Pole Angular Velocity Over Time')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Neural Network Policies

Instead of hardcoding policies, we can use neural networks to learn optimal actions. The network:
- Takes observations as input
- Outputs action probabilities
- Uses random selection based on probabilities (for exploration)

For CartPole:
- Input: 4 features (cart position, cart velocity, pole angle, pole angular velocity)
- Output: 1 value with sigmoid activation (probability of action 0)
- Action 1 has probability = 1 - p

In [ ]:
# Install TensorFlow if needed
!pip install tensorflow

In [ ]:
import tensorflow as tf
from tensorflow import keras

print(f"TensorFlow version: {tf.__version__}")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Create a neural network policy
n_inputs = 4  # Number of observation features

model = keras.models.Sequential([
    keras.layers.Dense(5, activation="elu", input_shape=[n_inputs]),
    keras.layers.Dense(1, activation="sigmoid"),
])

model.summary()

### Understanding the Neural Network Policy

The network outputs a probability value between 0 and 1:
- This value represents the probability of taking action 0 (left)
- Probability of action 1 (right) = 1 - p

**Why use random selection?**
- **Exploration**: Try new actions to discover better strategies
- **Exploitation**: Use known good actions more often
- This balance is crucial for learning optimal policies

In [ ]:
# Test the neural network policy
test_obs = np.array([[0.0, 0.0, 0.05, 0.0]])  # Batch of 1 observation

# Get probability of action 0 (left)
left_proba = model.predict(test_obs, verbose=0)
print(f"Observation: {test_obs[0]}")
print(f"Probability of action 0 (left): {left_proba[0, 0]:.4f}")
print(f"Probability of action 1 (right): {1 - left_proba[0, 0]:.4f}")

# Sample an action based on the probability
action = 0 if np.random.rand() < left_proba[0, 0] else 1
print(f"\nSampled action: {action}")

### Implementing a Stochastic Policy Function

Let's create a function that uses our neural network to select actions probabilistically.

In [ ]:
def neural_network_policy(model, obs):
    """
    Stochastic policy using a neural network.
    
    Args:
        model: Trained Keras model
        obs: Current observation
    
    Returns:
        action: Randomly selected action based on network output
    """
    left_proba = model.predict(obs[np.newaxis], verbose=0)[0, 0]
    action = 0 if np.random.rand() < left_proba else 1
    return action

# Test the policy multiple times with same observation
test_obs = np.array([0.0, 0.0, 0.05, 0.0])
actions = [neural_network_policy(model, test_obs) for _ in range(20)]

print(f"Test observation: {test_obs}")
print(f"Actions sampled (20 times): {actions}")
print(f"Action 0 frequency: {actions.count(0)/20:.2%}")
print(f"Action 1 frequency: {actions.count(1)/20:.2%}")

### Evaluating the Untrained Neural Network Policy

Let's see how our randomly initialized neural network performs compared to the basic policy.

In [ ]:
# Evaluate the untrained neural network policy
n_episodes = 100
max_steps = 200
nn_totals = []

for episode in range(n_episodes):
    episode_rewards = 0
    obs = env.reset()
    
    for step in range(max_steps):
        action = neural_network_policy(model, obs)
        obs, reward, done, info = env.step(action)
        episode_rewards += reward
        
        if done:
            break
    
    nn_totals.append(episode_rewards)

print(f"Untrained Neural Network Policy - {n_episodes} episodes")
print(f"Mean reward: {np.mean(nn_totals):.2f}")
print(f"Std reward: {np.std(nn_totals):.2f}")
print(f"\nComparison with Basic Policy:")
print(f"Basic policy mean: {np.mean(totals):.2f}")
print(f"Neural network (untrained) mean: {np.mean(nn_totals):.2f}")

## Summary of Part 1

In this first part, we've covered:

1. **Reinforcement Learning Basics**
   - Agents observe environments, take actions, and receive rewards
   - Goal: Maximize cumulative rewards over time

2. **Policy Search Approaches**
   - Brute force, genetic algorithms, and policy gradients
   - Policies can be deterministic or stochastic

3. **OpenAI Gym**
   - Created and explored the CartPole environment
   - Understanding observations, actions, and rewards

4. **Simple Policies**
   - Implemented a basic hard-coded policy using pole angle
   - Evaluated performance over multiple episodes

5. **Neural Network Policies**
   - Built a neural network to output action probabilities
   - Understood the importance of stochastic action selection for exploration

**Next Steps**: In the next part, we'll learn how to train these neural networks using Policy Gradients, including:
- The credit assignment problem
- Computing returns and advantages
- The REINFORCE algorithm
- Training loop implementation

In [ ]:
# Clean up
env.close()
print("Environment closed successfully!")

---

## Part 2: Policy Gradients and REINFORCE Algorithm

Now that we understand the basics of RL and have seen simple policies, let's learn how to train neural network policies using the REINFORCE algorithm.

## The Credit Assignment Problem

**Challenge**: When rewards are delayed, how do we know which actions were good?

For example, in a game:
- You take 100 actions during an episode
- You only get a reward at the end (win or lose)
- Which of the 100 actions contributed to winning/losing?

**Solution**: Evaluate actions based on the **return** - the sum of all discounted future rewards.

### Discount Factor (γ)

The discount factor determines how much we value future rewards compared to immediate rewards:
- **Return** = r₀ + γ·r₁ + γ²·r₂ + γ³·r₃ + ...
- **γ = 0**: Only care about immediate rewards (myopic)
- **γ = 1**: All future rewards equally important (far-sighted)
- **Typical values**: 0.9 to 0.99

This helps solve the credit assignment problem by giving more credit to actions that led to near-term rewards.

In [ ]:
# Demonstrate discount factor effect
def calculate_returns(rewards, discount_factor):
    """
    Calculate discounted returns for a sequence of rewards.
    
    Args:
        rewards: List of rewards [r0, r1, r2, ...]
        discount_factor: Gamma value (0 to 1)
    
    Returns:
        List of returns for each time step
    """
    returns = []
    G = 0  # Initialize return
    
    # Calculate returns backward from the end
    for reward in reversed(rewards):
        G = reward + discount_factor * G
        returns.insert(0, G)
    
    return returns

# Example: Episode with rewards [1, 1, 1, 1, 10]
rewards = [1, 1, 1, 1, 10]
print("Rewards at each step:", rewards)
print()

# Try different discount factors
for gamma in [0.0, 0.5, 0.9, 0.99, 1.0]:
    returns = calculate_returns(rewards, gamma)
    print(f"γ = {gamma:.2f}")
    print(f"  Returns: {[f'{r:.2f}' for r in returns]}")
    print(f"  Return at t=0: {returns[0]:.2f}")
    print()

### Visualizing Discount Factors

Let's visualize how different discount factors affect the value of future rewards.

In [ ]:
# Visualize how discount factor affects future reward values
steps = np.arange(0, 10)
gammas = [0.9, 0.95, 0.99]

plt.figure(figsize=(10, 5))

for gamma in gammas:
    discounts = [gamma**t for t in steps]
    plt.plot(steps, discounts, marker='o', label=f'γ = {gamma}')

plt.xlabel('Time Steps in Future')
plt.ylabel('Discount Multiplier (γ^t)')
plt.title('Effect of Discount Factor on Future Rewards')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.3, label='50% value')
plt.show()

print("Interpretation:")
print("- Lower γ (e.g., 0.9): Future rewards lose value quickly")
print("- Higher γ (e.g., 0.99): Future rewards maintain value longer")
print("- Agent with γ=0.9 is more 'impatient' than agent with γ=0.99")

## The REINFORCE Algorithm (Policy Gradients)

The REINFORCE algorithm (Williams, 1992) is a policy gradient method that:
1. Uses a neural network to represent the policy
2. Plays multiple episodes to collect experience
3. Updates the network to make good actions more likely

### Key Idea

- **Good actions** (high returns) → Increase their probability
- **Bad actions** (low returns) → Decrease their probability

### Algorithm Steps

1. **Play episodes**: Let the neural network play several episodes
2. **Compute gradients**: At each step, compute gradients that would make the chosen action more likely (but don't apply them yet)
3. **Compute advantages**: Calculate how good each action was (normalized discounted returns)
4. **Weight gradients**: Multiply gradients by advantages
   - Positive advantage → Apply gradients (make action more likely)
   - Negative advantage → Apply opposite gradients (make action less likely)
5. **Update policy**: Take the mean of all weighted gradients and perform Gradient Descent

### Implementation: Playing One Step

This function plays one step in the environment and computes gradients for the chosen action.

In [ ]:
def play_one_step(env, obs, model, loss_fn):
    """
    Play one step and compute gradients for the chosen action.
    
    Args:
        env: Gym environment
        obs: Current observation
        model: Neural network policy
        loss_fn: Loss function (binary crossentropy)
    
    Returns:
        obs: Next observation
        reward: Reward received
        done: Episode done flag
        grads: Gradients for this step
    """
    with tf.GradientTape() as tape:
        # Get probability of action 0 (left)
        left_proba = model(obs[np.newaxis])
        
        # Sample action: 1 if random > left_proba, else 0
        action = (tf.random.uniform([1, 1]) > left_proba)
        
        # Target: what we want the network to output
        # If action=0, target=1.0; if action=1, target=0.0
        y_target = tf.constant([[1.]]) - tf.cast(action, tf.float32)
        
        # Compute loss (how much we want to adjust the network)
        loss = tf.reduce_mean(loss_fn(y_target, left_proba))
    
    # Compute gradients
    grads = tape.gradient(loss, model.trainable_variables)
    
    # Take action in environment
    obs, reward, done, info = env.step(int(action[0, 0].numpy()))
    
    return obs, reward, done, grads

print("play_one_step function defined!")
print("\nThis function:")
print("1. Uses the model to get action probabilities")
print("2. Samples an action randomly based on probabilities")
print("3. Computes gradients that would make the chosen action more likely")
print("4. Takes the action in the environment")
print("5. Returns the results and gradients (not applied yet!)")

### Implementation: Playing Multiple Episodes

This function plays multiple complete episodes and collects all rewards and gradients.

In [ ]:
def play_multiple_episodes(env, n_episodes, n_max_steps, model, loss_fn):
    """
    Play multiple episodes and collect all rewards and gradients.
    
    Args:
        env: Gym environment
        n_episodes: Number of episodes to play
        n_max_steps: Maximum steps per episode
        model: Neural network policy
        loss_fn: Loss function
    
    Returns:
        all_rewards: List of reward lists (one per episode)
        all_grads: List of gradient lists (one per episode)
    """
    all_rewards = []
    all_grads = []
    
    for episode in range(n_episodes):
        current_rewards = []
        current_grads = []
        obs, info = env.reset()
        
        for step in range(n_max_steps):
            obs, reward, done, grads = play_one_step(env, obs, model, loss_fn)
            current_rewards.append(reward)
            current_grads.append(grads)
            
            if done:
                break
        
        all_rewards.append(current_rewards)
        all_grads.append(current_grads)
    
    return all_rewards, all_grads

print("play_multiple_episodes function defined!")
print("\nThis function:")
print("1. Plays n_episodes complete episodes")
print("2. Collects rewards and gradients for each step")
print("3. Stops an episode if done=True or max steps reached")
print("4. Returns all collected data for training")

### Implementation: Discount and Normalize Rewards

These functions compute discounted returns and normalize them to get advantages.

In [ ]:
def discount_rewards(rewards, discount_factor):
    """
    Compute discounted returns for a single episode.
    
    Args:
        rewards: List of rewards [r0, r1, r2, ...]
        discount_factor: Gamma value
    
    Returns:
        Array of discounted returns for each step
    """
    discounted = np.array(rewards)
    # Work backwards from the end
    for step in range(len(rewards) - 2, -1, -1):
        discounted[step] += discounted[step + 1] * discount_factor
    return discounted

def discount_and_normalize_rewards(all_rewards, discount_factor):
    """
    Discount and normalize rewards from multiple episodes.
    
    Args:
        all_rewards: List of reward lists (one per episode)
        discount_factor: Gamma value
    
    Returns:
        List of normalized discounted returns (advantages)
    """
    # Discount rewards for each episode
    all_discounted_rewards = [discount_rewards(rewards, discount_factor)
                               for rewards in all_rewards]
    
    # Flatten all discounted rewards
    flat_rewards = np.concatenate(all_discounted_rewards)
    
    # Normalize: subtract mean and divide by std
    reward_mean = flat_rewards.mean()
    reward_std = flat_rewards.std()
    
    # Return normalized discounted rewards
    return [(discounted_rewards - reward_mean) / reward_std
            for discounted_rewards in all_discounted_rewards]

print("Reward processing functions defined!")
print("\nNormalization benefits:")
print("- Positive values → actions were better than average")
print("- Negative values → actions were worse than average")
print("- Zero mean and unit variance → stable training")

### Testing Reward Processing

Let's test our reward processing functions with a simple example.

In [ ]:
# Test with two episodes
episode1_rewards = [1, 1, 1, 1, 1]  # Short episode, ended early
episode2_rewards = [1, 1, 1, 1, 1, 1, 1, 1, 1, 10]  # Long episode, big reward at end

all_rewards = [episode1_rewards, episode2_rewards]
discount_factor = 0.95

print("Original rewards:")
print(f"Episode 1: {episode1_rewards}")
print(f"Episode 2: {episode2_rewards}")
print()

# Discount rewards
discounted = [discount_rewards(rewards, discount_factor) for rewards in all_rewards]
print("Discounted returns (γ=0.95):")
for i, returns in enumerate(discounted):
    print(f"Episode {i+1}: {[f'{r:.2f}' for r in returns]}")
print()

# Normalize
normalized = discount_and_normalize_rewards(all_rewards, discount_factor)
print("Normalized discounted returns (advantages):")
for i, advantages in enumerate(normalized):
    print(f"Episode {i+1}: {[f'{a:.2f}' for a in advantages]}")
print()

print("Interpretation:")
print("- Episode 1 has negative advantages (performed worse than average)")
print("- Episode 2 has positive advantages (performed better than average)")
print("- The big reward at the end of episode 2 propagates backwards")

## Training the Policy with REINFORCE

Now we'll put it all together and train our neural network policy using the REINFORCE algorithm!

In [ ]:
# Create a fresh environment and model for training
env = gym.make("CartPole-v1")

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Create the model
n_inputs = 4
model = keras.models.Sequential([
    keras.layers.Dense(5, activation="elu", input_shape=[n_inputs]),
    keras.layers.Dense(1, activation="sigmoid"),
])

print("Environment and model created!")
print(f"Model has {model.count_params()} trainable parameters")

In [ ]:
# Training hyperparameters
n_iterations = 150              # Number of training iterations
n_episodes_per_update = 10      # Episodes to play before each update
n_max_steps = 200               # Maximum steps per episode
discount_factor = 0.95          # Gamma for discounting future rewards

# Create optimizer and loss function
optimizer = keras.optimizers.Adam(learning_rate=0.01)
loss_fn = keras.losses.binary_crossentropy

print("Training Configuration:")
print(f"  Iterations: {n_iterations}")
print(f"  Episodes per update: {n_episodes_per_update}")
print(f"  Max steps per episode: {n_max_steps}")
print(f"  Discount factor (γ): {discount_factor}")
print(f"  Learning rate: 0.01")
print()
print(f"Total episodes to play: {n_iterations * n_episodes_per_update}")

### The Training Loop

This is the core REINFORCE algorithm implementation:

In [ ]:
# Track rewards for plotting
all_iteration_rewards = []

print("Starting training...")
print("This may take a few minutes...\n")

for iteration in range(n_iterations):
    # Step 1: Play multiple episodes and collect data
    all_rewards, all_grads = play_multiple_episodes(
        env, n_episodes_per_update, n_max_steps, model, loss_fn)
    
    # Step 2: Compute advantages (normalized discounted returns)
    all_final_rewards = discount_and_normalize_rewards(all_rewards, discount_factor)
    
    # Step 3: Compute mean gradients weighted by advantages
    all_mean_grads = []
    for var_index in range(len(model.trainable_variables)):
        mean_grads = tf.reduce_mean(
            [final_reward * all_grads[episode_index][step][var_index]
             for episode_index, final_rewards in enumerate(all_final_rewards)
             for step, final_reward in enumerate(final_rewards)], axis=0)
        all_mean_grads.append(mean_grads)
    
    # Step 4: Apply gradients to update the policy
    optimizer.apply_gradients(zip(all_mean_grads, model.trainable_variables))
    
    # Track performance
    total_rewards = sum(map(sum, all_rewards))
    mean_reward = total_rewards / n_episodes_per_update
    all_iteration_rewards.append(mean_reward)
    
    # Print progress
    if iteration % 10 == 0:
        print(f"Iteration {iteration:3d}: Mean reward = {mean_reward:.2f}")

print("\nTraining complete!")

### Visualizing Training Progress

Let's see how the agent's performance improved over training:

In [ ]:
# Plot training progress
plt.figure(figsize=(14, 5))

# Plot 1: Raw rewards
plt.subplot(1, 2, 1)
plt.plot(all_iteration_rewards, alpha=0.6)
# Add moving average
window = 10
moving_avg = np.convolve(all_iteration_rewards, np.ones(window)/window, mode='valid')
plt.plot(range(window-1, len(all_iteration_rewards)), moving_avg, 'r-', linewidth=2, label='Moving Average')
plt.axhline(y=200, color='g', linestyle='--', label='Target (200)')
plt.xlabel('Iteration')
plt.ylabel('Mean Reward')
plt.title('Training Progress: Mean Reward per Iteration')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Success rate (rewards >= 190)
plt.subplot(1, 2, 2)
success_threshold = 190
successes = [1 if r >= success_threshold else 0 for r in all_iteration_rewards]
success_rate = np.convolve(successes, np.ones(window)/window, mode='valid')
plt.plot(range(window-1, len(successes)), success_rate * 100)
plt.xlabel('Iteration')
plt.ylabel('Success Rate (%)')
plt.title(f'Success Rate (Reward ≥ {success_threshold})')
plt.grid(True, alpha=0.3)
plt.ylim([0, 105])

plt.tight_layout()
plt.show()

print(f"\nFinal 10 iterations - Mean reward: {np.mean(all_iteration_rewards[-10:]):.2f}")
print(f"Best iteration reward: {np.max(all_iteration_rewards):.2f}")

### Testing the Trained Policy

Let's test our trained policy and compare it to the original untrained and basic policies:

In [ ]:
# Test the trained model
n_test_episodes = 100
test_rewards = []

for episode in range(n_test_episodes):
    obs, info = env.reset()
    episode_reward = 0
    
    for step in range(200):
        # Use trained model to select actions
        left_proba = model.predict(obs[np.newaxis], verbose=0)[0, 0]
        action = 0 if np.random.rand() < left_proba else 1
        
        obs, reward, done, info = env.step(action)
        episode_reward += reward
        
        if done:
            break
    
    test_rewards.append(episode_reward)

print("Performance Comparison:")
print("=" * 50)
print(f"Basic Policy (hardcoded):        {np.mean(totals):.2f} ± {np.std(totals):.2f}")
print(f"Trained Policy (REINFORCE):      {np.mean(test_rewards):.2f} ± {np.std(test_rewards):.2f}")
print()
print(f"Improvement: {np.mean(test_rewards) - np.mean(totals):.2f} points")
print(f"Success rate (≥195): {sum(1 for r in test_rewards if r >= 195) / n_test_episodes * 100:.1f}%")

In [ ]:
# Visualize test results
plt.figure(figsize=(12, 5))

# Distribution comparison
plt.subplot(1, 2, 1)
plt.hist(totals, bins=30, alpha=0.5, label='Basic Policy', edgecolor='black')
plt.hist(test_rewards, bins=30, alpha=0.5, label='Trained Policy', edgecolor='black')
plt.xlabel('Episode Reward')
plt.ylabel('Frequency')
plt.title('Reward Distribution Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

# Cumulative distribution
plt.subplot(1, 2, 2)
sorted_basic = np.sort(totals)
sorted_trained = np.sort(test_rewards)
plt.plot(sorted_basic, np.linspace(0, 1, len(sorted_basic)), label='Basic Policy')
plt.plot(sorted_trained, np.linspace(0, 1, len(sorted_trained)), label='Trained Policy')
plt.xlabel('Episode Reward')
plt.ylabel('Cumulative Probability')
plt.title('Cumulative Distribution Function')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary of Parts 1-2

### What We've Learned

1. **Reinforcement Learning Fundamentals**
   - Agents interact with environments through observations, actions, and rewards
   - Goal: Learn a policy that maximizes cumulative rewards

2. **Policies**
   - **Deterministic**: Same action for same state (basic policy)
   - **Stochastic**: Probabilistic action selection (neural network policy)

3. **The Credit Assignment Problem**
   - Challenge: Which actions led to good/bad outcomes?
   - Solution: Discounted returns (γ = 0.9-0.99)

4. **REINFORCE Algorithm (Policy Gradients)**
   - Play episodes and collect gradients
   - Compute advantages (normalized discounted returns)
   - Weight gradients by advantages
   - Update policy to make good actions more likely

5. **Training Results**
   - Successfully trained a neural network to balance CartPole
   - Significant improvement over basic hardcoded policy
   - Demonstrated the power of learning from experience

### Key Insights

✅ **Exploration is crucial**: Stochastic policies explore different actions  
✅ **Rewards must be processed**: Discounting and normalization stabilize learning  
✅ **Gradients are weighted**: Good actions get reinforced, bad actions get discouraged  
✅ **Sample inefficiency**: Policy gradients need many episodes to learn  

### Limitations

⚠️ **Sample inefficiency**: REINFORCE requires many episodes  
⚠️ **High variance**: Performance can vary significantly  
⚠️ **Sensitive to hyperparameters**: Learning rate, discount factor, etc.  

**Next topics** (for future parts): Markov Decision Processes, Q-Learning, Deep Q-Networks, and advanced algorithms!

In [ ]:
# Final cleanup
env.close()
print("✓ Environment closed")
print("✓ Notebook complete!")
print("\nYou've successfully implemented and trained a reinforcement learning agent!")
print("The agent learned to balance a pole on a cart using only trial and error.")

---

## Part 3: Markov Decision Processes (MDPs) and Q-Learning

Now we'll explore a more formal mathematical framework for RL and introduce value-based methods.

## Markov Decision Processes (MDPs)

**Markov Chain**: A stochastic process with fixed states that randomly evolves between states with fixed probabilities, depending only on the current state (no memory of past states).

**Markov Decision Process (MDP)**: An extension of Markov chains where:
- An agent can choose from several **actions** at each step
- **Transition probabilities** depend on the chosen action
- Some transitions return **rewards**
- **Goal**: Find a policy that maximizes rewards over time

### Key Components of an MDP

1. **States (S)**: Set of possible states
2. **Actions (A)**: Set of possible actions  
3. **Transition function T(s,a,s')**: Probability of transitioning from state s to s' when taking action a
4. **Reward function R(s,a,s')**: Reward received for transition from s to s' via action a
5. **Discount factor (γ)**: How much we value future rewards (0 to 1)
6. **Policy (π)**: Strategy for choosing actions

### Example MDP: Simple Grid World

Let's create a simple 3-state MDP to understand the concepts:

**States**: 
- State 0: Start state
- State 1: Middle state  
- State 2: Goal state

**Actions**:
- State 0: can take actions {0, 1, 2}
- State 1: can take actions {0, 2}
- State 2: can take action {1}

**Rewards**:
- Reaching state 0 from state 0 via action 0: +10
- Reaching state 0 from state 2 via action 1: +40
- Reaching state 2 from state 1 via action 2: -50
- All other transitions: 0

In [ ]:
# Define the MDP
# Transition probabilities: transition_probabilities[s][a][s'] = P(s'|s,a)
transition_probabilities = [
    # State 0: 3 actions
    [[0.7, 0.3, 0.0],   # Action 0: 70% stay in state 0, 30% to state 1
     [1.0, 0.0, 0.0],   # Action 1: 100% stay in state 0
     [0.8, 0.2, 0.0]],  # Action 2: 80% stay in state 0, 20% to state 1
    
    # State 1: 2 actions (action 1 not available)
    [[0.0, 1.0, 0.0],   # Action 0: 100% stay in state 1
     None,              # Action 1 not available
     [0.0, 0.0, 1.0]],  # Action 2: 100% go to state 2
    
    # State 2: 1 action (only action 1 available)
    [None,              # Action 0 not available
     [0.8, 0.1, 0.1],   # Action 1: 80% to state 0, 10% to state 1, 10% stay
     None]              # Action 2 not available
]

# Rewards: rewards[s][a][s'] = reward for transitioning from s to s' via action a
rewards = [
    [[+10, 0, 0],    # State 0, action 0
     [0, 0, 0],      # State 0, action 1
     [0, 0, 0]],     # State 0, action 2
    
    [[0, 0, 0],      # State 1, action 0
     [0, 0, 0],      # State 1, action 1 (not available)
     [0, 0, -50]],   # State 1, action 2
    
    [[0, 0, 0],      # State 2, action 0 (not available)
     [+40, 0, 0],    # State 2, action 1
     [0, 0, 0]]      # State 2, action 2 (not available)
]

# Available actions for each state
possible_actions = [[0, 1, 2], [0, 2], [1]]

print("MDP Defined!")
print(f"Number of states: {len(possible_actions)}")
print(f"Available actions per state: {possible_actions}")
print("\nKey transitions:")
print("  State 0, Action 0 → State 0 (70%): Reward +10")
print("  State 2, Action 1 → State 0 (80%): Reward +40")
print("  State 1, Action 2 → State 2 (100%): Reward -50")

## Bellman Optimality Equation

The **optimal state value** V*(s) represents the maximum expected return starting from state s:

**V*(s) = max_a Σ_{s'} T(s,a,s')[R(s,a,s') + γ·V*(s')]**

Where:
- **T(s,a,s')**: Transition probability from s to s' given action a
- **R(s,a,s')**: Reward for transition s→s' with action a  
- **γ**: Discount factor
- **V*(s')**: Optimal value of next state

This equation says: "The value of a state is the maximum expected immediate reward plus the discounted value of the next state."

## Q-Value Iteration

Instead of tracking state values V(s), we can track **Q-values** (quality values) for state-action pairs Q(s,a).

**Q*(s,a)** represents the expected return from taking action a in state s, then following the optimal policy.

**Q-Value Iteration Update Rule**:

**Q_{k+1}(s,a) ← Σ_{s'} T(s,a,s')[R(s,a,s') + γ·max_{a'} Q_k(s',a')]**

**Optimal Policy**: π*(s) = argmax_a Q*(s,a) - choose the action with highest Q-value!

### Advantages of Q-Values
- No need to know transition probabilities when selecting actions
- Just pick the action with the highest Q-value
- This makes Q-values perfect for model-free RL

In [ ]:
# Initialize Q-Values
# Q_values[s, a] represents the Q-value for state s, action a
Q_values = np.full((3, 3), -np.inf)  # Start with -inf for impossible actions

# Set initial Q-values to 0 for possible actions
for state, actions in enumerate(possible_actions):
    Q_values[state, actions] = 0.0

print("Initial Q-Values:")
print(Q_values)
print("\nNote: -inf values indicate actions that are not available in that state")

In [ ]:
# Run Q-Value Iteration
gamma = 0.90  # Discount factor
n_iterations = 50

print("Running Q-Value Iteration...")
print("=" * 60)

for iteration in range(n_iterations):
    Q_prev = Q_values.copy()
    
    # Update Q-value for each state-action pair
    for s in range(3):  # For each state
        for a in possible_actions[s]:  # For each possible action in this state
            # Compute Q(s,a) using the Bellman equation
            Q_values[s, a] = np.sum([
                transition_probabilities[s][a][sp]  # P(s'|s,a)
                * (rewards[s][a][sp] + gamma * np.max(Q_prev[sp]))  # R + γ·max Q(s',a')
                for sp in range(3)  # Sum over all possible next states
            ])
    
    # Print progress every 10 iterations
    if iteration % 10 == 0:
        print(f"Iteration {iteration:2d}:")
        print(Q_values)
        print()

print("Final Q-Values after convergence:")
print(Q_values)
print("\n" + "=" * 60)

In [ ]:
# Extract the optimal policy
print("Optimal Policy (best action for each state):")
print("=" * 60)

for state in range(3):
    # Find the action with maximum Q-value
    best_action = np.argmax(Q_values[state])
    best_q_value = Q_values[state, best_action]
    
    print(f"\nState {state}:")
    print(f"  Available actions: {possible_actions[state]}")
    print(f"  Q-values: {Q_values[state, possible_actions[state]]}")
    print(f"  Best action: {best_action} (Q-value: {best_q_value:.2f})")

print("\n" + "=" * 60)
print("\nOptimal Policy Summary:")
print("  State 0 → Action", np.argmax(Q_values[0]))
print("  State 1 → Action", np.argmax(Q_values[1]))
print("  State 2 → Action", np.argmax(Q_values[2]))

### Visualizing Q-Value Convergence

Let's visualize how Q-values converge over iterations:

In [ ]:
# Re-run Q-Value Iteration and track history
Q_values = np.full((3, 3), -np.inf)
for state, actions in enumerate(possible_actions):
    Q_values[state, actions] = 0.0

# Track Q-value history
history = {f"S{s}_A{a}": [] for s in range(3) for a in possible_actions[s]}

for iteration in range(50):
    Q_prev = Q_values.copy()
    
    for s in range(3):
        for a in possible_actions[s]:
            Q_values[s, a] = np.sum([
                transition_probabilities[s][a][sp]
                * (rewards[s][a][sp] + gamma * np.max(Q_prev[sp]))
                for sp in range(3)
            ])
            history[f"S{s}_A{a}"].append(Q_values[s, a])

# Plot convergence
plt.figure(figsize=(14, 5))

# Plot Q-values for State 0
plt.subplot(1, 3, 1)
for a in possible_actions[0]:
    plt.plot(history[f"S0_A{a}"], label=f"Action {a}")
plt.xlabel('Iteration')
plt.ylabel('Q-Value')
plt.title('State 0: Q-Value Convergence')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot Q-values for State 1
plt.subplot(1, 3, 2)
for a in possible_actions[1]:
    plt.plot(history[f"S1_A{a}"], label=f"Action {a}")
plt.xlabel('Iteration')
plt.ylabel('Q-Value')
plt.title('State 1: Q-Value Convergence')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot Q-values for State 2
plt.subplot(1, 3, 3)
for a in possible_actions[2]:
    plt.plot(history[f"S2_A{a}"], label=f"Action {a}")
plt.xlabel('Iteration')
plt.ylabel('Q-Value')
plt.title('State 2: Q-Value Convergence')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice how Q-values converge to stable values over iterations!")

## Q-Learning: Learning from Experience

**Problem with Q-Value Iteration**: It requires knowing the transition probabilities and rewards in advance (model-based).

**Q-Learning Solution**: Learn Q-values by interacting with the environment (model-free)!

### Q-Learning Update Rule

**Q(s,a) ← (1-α)Q(s,a) + α[r + γ·max_{a'} Q(s',a')]**

Or equivalently:

**Q(s,a) ← Q(s,a) + α·δ**

Where **δ = r + γ·max_{a'} Q(s',a') - Q(s,a)** is the **TD (Temporal Difference) error**

### Key Parameters
- **α (alpha)**: Learning rate - how much to update Q-values (typically 0.01 to 0.1)
- **γ (gamma)**: Discount factor - how much to value future rewards (typically 0.9 to 0.99)

### Key Characteristics
- **Off-policy**: The policy being learned (greedy) differs from the exploration policy (ε-greedy)
- **Model-free**: Doesn't need to know transition probabilities
- **Learns from experience**: Improves Q-values through trial and error

### Implementing Q-Learning

Let's implement Q-Learning for our simple MDP:

In [ ]:
# Helper functions for Q-Learning
def step(state, action):
    """
    Simulate taking an action in the environment.
    
    Args:
        state: Current state
        action: Action to take
    
    Returns:
        next_state: Resulting state
        reward: Reward received
    """
    probas = transition_probabilities[state][action]
    next_state = np.random.choice([0, 1, 2], p=probas)
    reward = rewards[state][action][next_state]
    return next_state, reward

def exploration_policy(state):
    """
    Random exploration policy - choose any available action randomly.
    
    Args:
        state: Current state
    
    Returns:
        action: Randomly chosen action
    """
    return np.random.choice(possible_actions[state])

print("Helper functions defined!")
print("\nstep(state, action): Simulates environment dynamics")
print("exploration_policy(state): Returns random action for exploration")

In [ ]:
# Initialize Q-values for Q-Learning
Q_values_ql = np.full((3, 3), -np.inf)
for state, actions in enumerate(possible_actions):
    Q_values_ql[state, actions] = 0.0

# Q-Learning hyperparameters
alpha0 = 0.05      # Initial learning rate
decay = 0.005      # Learning rate decay
gamma = 0.90       # Discount factor
n_iterations = 10000

# Track Q-values for visualization
q_history = []
state = 0  # Start in state 0

print("Starting Q-Learning...")
print(f"Parameters: α₀={alpha0}, decay={decay}, γ={gamma}")
print(f"Iterations: {n_iterations}")
print()

for iteration in range(n_iterations):
    # Choose action using exploration policy
    action = exploration_policy(state)
    
    # Take action and observe result
    next_state, reward = step(state, action)
    
    # Find maximum Q-value for next state
    next_value = np.max(Q_values_ql[next_state])
    
    # Calculate learning rate (decays over time)
    alpha = alpha0 / (1 + iteration * decay)
    
    # Q-Learning update
    Q_values_ql[state, action] = (1 - alpha) * Q_values_ql[state, action] + \
                                  alpha * (reward + gamma * next_value)
    
    # Move to next state
    state = next_state
    
    # Track history every 100 iterations
    if iteration % 100 == 0:
        q_history.append(Q_values_ql.copy())
    
    # Print progress
    if iteration % 2000 == 0:
        print(f"Iteration {iteration:5d}: α={alpha:.4f}")
        print(Q_values_ql)
        print()

print("\nQ-Learning Complete!")
print("Final Q-Values:")
print(Q_values_ql)

In [ ]:
# Compare Q-Learning results with Q-Value Iteration
print("Comparison: Q-Value Iteration vs Q-Learning")
print("=" * 60)
print("\nQ-Value Iteration (model-based - knows MDP dynamics):")
print(Q_values)
print("\nQ-Learning (model-free - learns from experience):")
print(Q_values_ql)
print("\nDifference:")
# Only compare available actions
diff = np.full((3, 3), np.nan)
for s in range(3):
    for a in possible_actions[s]:
        diff[s, a] = abs(Q_values[s, a] - Q_values_ql[s, a])
print(diff)
print("\nNote: Q-Learning converges to similar values without knowing the MDP!")

### Exploration Strategies

**Problem**: Random exploration can be inefficient.

**Solution 1: ε-greedy Policy**
- With probability ε: Take random action (explore)
- With probability 1-ε: Take best action (exploit)
- Start with high ε (e.g., 1.0), gradually decrease to low value (e.g., 0.05)

**Solution 2: Exploration Function**
- Add curiosity bonus to encourage trying unexplored actions
- Q(s,a) + κ/(1+N(s,a)) where N(s,a) = number of times action a was taken in state s
- Agent is "curious" about actions it hasn't tried much

In [ ]:
# Implement ε-greedy policy
def epsilon_greedy_policy(state, epsilon=0.1):
    """
    ε-greedy exploration policy.
    
    Args:
        state: Current state
        epsilon: Probability of random action
    
    Returns:
        action: Selected action
    """
    if np.random.rand() < epsilon:
        # Explore: random action
        return np.random.choice(possible_actions[state])
    else:
        # Exploit: best action
        return possible_actions[state][np.argmax(Q_values_ql[state, possible_actions[state]])]

# Test the policy
print("Testing ε-greedy policy in State 0:")
print(f"Q-values for State 0: {Q_values_ql[0, possible_actions[0]]}")
print(f"Best action: {possible_actions[0][np.argmax(Q_values_ql[0, possible_actions[0]])]}")
print()

# Sample actions 20 times with different epsilon values
for epsilon in [0.0, 0.1, 0.5, 1.0]:
    actions = [epsilon_greedy_policy(0, epsilon) for _ in range(100)]
    action_counts = {a: actions.count(a) for a in possible_actions[0]}
    print(f"ε={epsilon:.1f}: {action_counts} (out of 100 samples)")

## Summary of Part 3

In this section, we covered:

### 1. **Markov Decision Processes (MDPs)**
   - Formal framework for sequential decision making
   - Components: States, Actions, Transitions, Rewards, Discount factor
   - Bellman Optimality Equation

### 2. **Q-Value Iteration (Model-Based)**
   - Iteratively compute optimal Q-values
   - Requires knowing transition probabilities and rewards
   - Converges to optimal policy

### 3. **Q-Learning (Model-Free)**
   - Learn Q-values from experience without knowing the MDP
   - Update rule: Q(s,a) ← Q(s,a) + α[r + γ·max Q(s',a') - Q(s,a)]
   - Off-policy learning

### 4. **Exploration Strategies**
   - ε-greedy: Balance exploration and exploitation
   - Exploration functions: Add curiosity bonus

### Key Insights
- **Model-based vs Model-free**: Q-Value Iteration needs MDP knowledge, Q-Learning doesn't
- **Sample efficiency**: Model-based methods are more sample-efficient but less flexible
- **Exploration is crucial**: Need to try different actions to learn optimal policy

**Next**: We'll apply Q-Learning to CartPole using Deep Neural Networks (Deep Q-Networks)!

---

## Part 4: Deep Q-Networks (DQN)

For environments with large state spaces (like images), we can't store Q-values in a table. Instead, we use a neural network to approximate Q-values!

## Deep Q-Learning Concept

**Problem**: CartPole has 4 continuous state variables → infinite possible states → can't use Q-table

**Solution**: Use a Deep Neural Network to approximate Q-values: **Q_θ(s,a)**

### How DQN Works

1. **Input**: Current state observation (e.g., [cart_pos, cart_vel, pole_angle, pole_vel])
2. **Output**: Q-values for all possible actions [Q(s,a₀), Q(s,a₁), ...]
3. **Training**: Minimize difference between predicted and target Q-values

### Target Q-Value

**Q_target(s,a) = r + γ·max_{a'} Q_θ(s',a')**

Train the DQN to minimize: **(Q_θ(s,a) - Q_target)²**

### Key Challenges

1. **Non-stationary targets**: Target Q-values change as we train
2. **Correlated experiences**: Consecutive states are highly correlated
3. **Catastrophic forgetting**: Agent forgets what it learned

### Solutions

1. **Replay Buffer**: Store experiences and sample randomly (breaks correlation)
2. **Target Network**: Use separate network for computing targets (stabilizes training)

### Building the DQN Model

Let's create a neural network for CartPole:

In [ ]:
# Create environment for DQN
env = gym.make("CartPole-v1")
input_shape = [4]  # State has 4 features
n_outputs = 2      # 2 possible actions

# Build the DQN model
model_dqn = keras.models.Sequential([
    keras.layers.Dense(32, activation="elu", input_shape=input_shape),
    keras.layers.Dense(32, activation="elu"),
    keras.layers.Dense(n_outputs)  # Output Q-value for each action
])

print("DQN Model Architecture:")
model_dqn.summary()
print("\nKey differences from Policy Gradient model:")
print("- Outputs Q-values for ALL actions (not action probabilities)")
print("- No sigmoid activation on output (Q-values can be any real number)")
print("- Will be trained using MSE loss (not cross-entropy)")

### ε-Greedy Policy for DQN

The policy uses the DQN to select actions:

In [ ]:
def epsilon_greedy_policy_dqn(state, epsilon=0):
    """
    ε-greedy policy using DQN.
    
    Args:
        state: Current state
        epsilon: Probability of random action
    
    Returns:
        action: Selected action
    """
    if np.random.rand() < epsilon:
        # Explore: random action
        return np.random.randint(n_outputs)
    else:
        # Exploit: action with highest Q-value
        Q_values = model_dqn.predict(state[np.newaxis], verbose=0)
        return np.argmax(Q_values[0])

# Test the policy
test_state, info = env.reset()
print("Testing ε-greedy policy with DQN:")
print(f"State: {test_state}")
print()

# Get Q-values
Q_vals = model_dqn.predict(test_state[np.newaxis], verbose=0)[0]
print(f"Q-values: {Q_vals}")
print(f"Best action (ε=0): {epsilon_greedy_policy_dqn(test_state, epsilon=0)}")
print()

# Sample with different epsilon values
for eps in [0.0, 0.5, 1.0]:
    actions = [epsilon_greedy_policy_dqn(test_state, epsilon=eps) for _ in range(100)]
    print(f"ε={eps}: Action 0: {actions.count(0)}%, Action 1: {actions.count(1)}%")

### Replay Buffer

The replay buffer stores experiences (state, action, reward, next_state, done) and allows us to sample random batches for training. This breaks the correlation between consecutive experiences.

In [ ]:
from collections import deque

# Create replay buffer
replay_buffer = deque(maxlen=2000)

def sample_experiences(batch_size):
    """
    Sample random batch of experiences from replay buffer.
    
    Args:
        batch_size: Number of experiences to sample
    
    Returns:
        Tuple of (states, actions, rewards, next_states, dones)
    """
    indices = np.random.randint(len(replay_buffer), size=batch_size)
    batch = [replay_buffer[index] for index in indices]
    
    # Unpack experiences
    states, actions, rewards, next_states, dones = [
        np.array([experience[field_index] for experience in batch])
        for field_index in range(5)
    ]
    
    return states, actions, rewards, next_states, dones

def play_one_step_dqn(env, state, epsilon):
    """
    Play one step and store experience in replay buffer.
    
    Args:
        env: Gym environment
        state: Current state
        epsilon: Exploration rate
    
    Returns:
        Tuple of (next_state, reward, done, info)
    """
    action = epsilon_greedy_policy_dqn(state, epsilon)
    next_state, reward, done, info = env.step(action)
    replay_buffer.append((state, action, reward, next_state, done))
    return next_state, reward, done, info

print("Replay Buffer Functions Created!")
print(f"Buffer capacity: {replay_buffer.maxlen}")
print(f"Current buffer size: {len(replay_buffer)}")
print("\nFunctions:")
print("  - play_one_step_dqn(): Play step and store experience")
print("  - sample_experiences(): Sample random batch for training")

### Training Step

The training step samples a batch from the replay buffer and updates the DQN:

In [ ]:
# Training hyperparameters
batch_size = 32
discount_factor = 0.95
optimizer_dqn = keras.optimizers.Adam(learning_rate=1e-3)
loss_fn_dqn = keras.losses.mean_squared_error

def training_step_dqn(batch_size):
    """
    Train the DQN on a batch of experiences.
    
    Args:
        batch_size: Size of batch to sample
    
    Returns:
        loss: Training loss value
    """
    # Sample random batch
    experiences = sample_experiences(batch_size)
    states, actions, rewards, next_states, dones = experiences
    
    # Compute target Q-values
    next_Q_values = model_dqn.predict(next_states, verbose=0)
    max_next_Q_values = np.max(next_Q_values, axis=1)
    target_Q_values = (rewards + 
                       (1 - dones) * discount_factor * max_next_Q_values)
    
    # Create mask to select Q-values for actions that were taken
    mask = tf.one_hot(actions, n_outputs)
    
    # Train the model
    with tf.GradientTape() as tape:
        all_Q_values = model_dqn(states)
        Q_values = tf.reduce_sum(all_Q_values * mask, axis=1, keepdims=True)
        loss = tf.reduce_mean(loss_fn_dqn(target_Q_values, Q_values))
    
    grads = tape.gradient(loss, model_dqn.trainable_variables)
    optimizer_dqn.apply_gradients(zip(grads, model_dqn.trainable_variables))
    
    return loss

print("Training function created!")
print("\nTraining process:")
print("1. Sample random batch from replay buffer")
print("2. Compute target Q-values: r + γ·max Q(s',a')")
print("3. Compute loss: MSE between predicted and target Q-values")
print("4. Update network weights using gradient descent")

### Training the DQN

Now let's train the agent! We'll use a decaying epsilon schedule to gradually shift from exploration to exploitation.

In [ ]:
# Training configuration
n_episodes = 300
training_start = 50  # Start training after collecting some experiences
episode_rewards = []

print("Training DQN Agent...")
print(f"Episodes: {n_episodes}")
print(f"Training starts after: {training_start} episodes")
print(f"Batch size: {batch_size}")
print("=" * 60)

for episode in range(n_episodes):
    obs, info = env.reset()
    episode_reward = 0
    
    # Epsilon decay: start at 1.0, end at 0.01
    epsilon = max(1 - episode / 500, 0.01)
    
    for step in range(200):
        obs, reward, done, info = play_one_step_dqn(env, obs, epsilon)
        episode_reward += reward
        
        if done:
            break
    
    # Store episode reward
    episode_rewards.append(episode_reward)
    
    # Train the model (only after collecting enough experiences)
    if episode > training_start:
        training_step_dqn(batch_size)
    
    # Print progress
    if episode % 20 == 0:
        mean_reward = np.mean(episode_rewards[-50:]) if len(episode_rewards) >= 50 else np.mean(episode_rewards)
        print(f"Episode {episode:3d}: Reward = {episode_reward:6.1f}, "
              f"Mean(50) = {mean_reward:6.2f}, ε = {epsilon:.3f}")

print("\n" + "=" * 60)
print("Training Complete!")
print(f"Final mean reward (last 50 episodes): {np.mean(episode_rewards[-50:]):.2f}")

### Visualizing DQN Training Progress

In [ ]:
# Plot DQN training results
plt.figure(figsize=(14, 5))

# Plot 1: Episode rewards with moving average
plt.subplot(1, 2, 1)
plt.plot(episode_rewards, alpha=0.4, label='Episode Reward')
window = 20
moving_avg = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
plt.plot(range(window-1, len(episode_rewards)), moving_avg, 'r-', linewidth=2, label='Moving Average (20)')
plt.axhline(y=195, color='g', linestyle='--', alpha=0.5, label='Success Threshold')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('DQN Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Epsilon decay
plt.subplot(1, 2, 2)
epsilons = [max(1 - ep / 500, 0.01) for ep in range(n_episodes)]
plt.plot(epsilons)
plt.xlabel('Episode')
plt.ylabel('Epsilon (ε)')
plt.title('Exploration Rate Decay')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistics
print("\nTraining Statistics:")
print(f"  Episodes: {n_episodes}")
print(f"  Mean reward (first 50): {np.mean(episode_rewards[:50]):.2f}")
print(f"  Mean reward (last 50): {np.mean(episode_rewards[-50:]):.2f}")
print(f"  Max reward: {np.max(episode_rewards):.2f}")
print(f"  Episodes with reward ≥ 195: {sum(1 for r in episode_rewards if r >= 195)}")

### Comparing All Three Approaches

Let's compare the three methods we've implemented:

In [ ]:
# Test all three trained models
print("Performance Comparison: All Methods")
print("=" * 70)

# Test DQN
dqn_test_rewards = []
for episode in range(50):
    obs, info = env.reset()
    ep_reward = 0
    for step in range(200):
        action = epsilon_greedy_policy_dqn(obs, epsilon=0.01)
        obs, reward, done, info = env.step(action)
        ep_reward += reward
        if done:
            break
    dqn_test_rewards.append(ep_reward)

# Display results
print(f"\n1. Basic Policy (hardcoded):         {np.mean(totals):.2f} ± {np.std(totals):.2f}")
print(f"2. REINFORCE (Policy Gradients):     {np.mean(test_rewards):.2f} ± {np.std(test_rewards):.2f}")
print(f"3. DQN (Deep Q-Learning):            {np.mean(dqn_test_rewards):.2f} ± {np.std(dqn_test_rewards):.2f}")

print("\n" + "=" * 70)

# Visualize comparison
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
methods = ['Basic\nPolicy', 'REINFORCE', 'DQN']
means = [np.mean(totals), np.mean(test_rewards), np.mean(dqn_test_rewards)]
stds = [np.std(totals), np.std(test_rewards), np.std(dqn_test_rewards)]
plt.bar(methods, means, yerr=stds, alpha=0.7, capsize=10)
plt.ylabel('Mean Reward')
plt.title('Performance Comparison')
plt.axhline(y=195, color='r', linestyle='--', alpha=0.5, label='Success (195)')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

plt.subplot(1, 2, 2)
plt.hist(totals, bins=20, alpha=0.5, label='Basic Policy', density=True)
plt.hist(test_rewards, bins=20, alpha=0.5, label='REINFORCE', density=True)
plt.hist(dqn_test_rewards, bins=20, alpha=0.5, label='DQN', density=True)
plt.xlabel('Episode Reward')
plt.ylabel('Density')
plt.title('Reward Distributions')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## DQN Improvements

While our basic DQN works, there are several improvements that can make it more stable and efficient:

### 1. Fixed Q-Value Targets

**Problem**: Target Q-values change during training, making learning unstable.

**Solution**: Use two networks:
- **Online model**: Updated at each step, used to select actions
- **Target model**: Clone of online model, used only for computing targets
- Update target model periodically (e.g., every 50 episodes)

### 2. Double DQN

**Problem**: DQN tends to overestimate Q-values.

**Solution**: 
- Use online model to **select** best actions for next states
- Use target model to **evaluate** Q-values of those actions
- Reduces overestimation bias

### 3. Dueling DQN

**Problem**: Sometimes the state value matters more than individual action advantages.

**Solution**: Split the network into two streams:
- **Value stream**: Estimates V(s) - how good is this state?
- **Advantage stream**: Estimates A(s,a) - how much better is action a than average?
- **Q(s,a) = V(s) + A(s,a) - mean(A(s,·))**

### 4. Prioritized Experience Replay

**Problem**: All experiences are equally likely to be sampled, but some are more important.

**Solution**:
- Priority p = |TD error| = |r + γ·max Q(s',a') - Q(s,a)|
- Sample experiences with higher TD errors more frequently
- Agent learns more from surprising experiences

### Implementing Fixed Q-Value Targets

Let's implement the target network improvement:

In [ ]:
# Create target network (clone of online network)
target_model = keras.models.clone_model(model_dqn)
target_model.set_weights(model_dqn.get_weights())

print("Target Network Created!")
print("\nTwo-Network Architecture:")
print("  - Online Model: Updated every training step, used for action selection")
print("  - Target Model: Updated every N episodes, used for computing target Q-values")
print("\nBenefit: Stabilizes training by keeping targets constant for multiple updates")

# Example of updating target network
def update_target_network():
    """Copy weights from online model to target model."""
    target_model.set_weights(model_dqn.get_weights())
    print("✓ Target network updated")

In [ ]:
# Modified training step using target network
def training_step_with_target(batch_size):
    """
    Train DQN using target network for stable Q-value targets.
    """
    experiences = sample_experiences(batch_size)
    states, actions, rewards, next_states, dones = experiences
    
    # Use TARGET network to compute target Q-values (more stable)
    next_Q_values = target_model.predict(next_states, verbose=0)
    max_next_Q_values = np.max(next_Q_values, axis=1)
    target_Q_values = (rewards + 
                       (1 - dones) * discount_factor * max_next_Q_values)
    
    mask = tf.one_hot(actions, n_outputs)
    
    # Train ONLINE network
    with tf.GradientTape() as tape:
        all_Q_values = model_dqn(states)
        Q_values = tf.reduce_sum(all_Q_values * mask, axis=1, keepdims=True)
        loss = tf.reduce_mean(loss_fn_dqn(target_Q_values, Q_values))
    
    grads = tape.gradient(loss, model_dqn.trainable_variables)
    optimizer_dqn.apply_gradients(zip(grads, model_dqn.trainable_variables))
    
    return loss

print("Improved training step created!")
print("\nKey change: Uses target_model for computing Q(s',a') instead of model_dqn")
print("This prevents the target from moving too quickly during training")

## Final Summary: Complete Reinforcement Learning Journey

Congratulations! You've implemented and understood multiple RL algorithms from scratch!

### What We've Covered

#### **Part 1: Foundations (0-25%)**
✅ Reinforcement Learning basics  
✅ OpenAI Gym environments  
✅ Observations, actions, and rewards  
✅ Simple hardcoded policies  

#### **Part 2: Policy Gradients (25-50%)**
✅ Neural network policies  
✅ Credit assignment problem  
✅ Discount factors and returns  
✅ REINFORCE algorithm (Policy Gradients)  
✅ Training with advantages  

#### **Part 3: Value-Based Methods (50-75%)**
✅ Markov Decision Processes (MDPs)  
✅ Bellman Optimality Equation  
✅ Q-Value Iteration (model-based)  
✅ Q-Learning (model-free)  
✅ Exploration strategies (ε-greedy)  

#### **Part 4: Deep Q-Networks (75-100%)**
✅ Deep Q-Networks (DQN)  
✅ Replay buffer  
✅ Training with experience replay  
✅ Fixed Q-value targets  
✅ DQN improvements (Double DQN, Dueling DQN, PER)  

### Key Insights

| Method | Type | Pros | Cons |
|--------|------|------|------|
| **Policy Gradients** | Policy-based | Direct policy optimization, works with continuous actions | Sample inefficient, high variance |
| **Q-Learning** | Value-based | Sample efficient, off-policy | Only discrete actions, requires exploration |
| **DQN** | Value-based + Deep Learning | Handles high-dimensional states, sample efficient | Complex, sensitive to hyperparameters |

### Performance Summary

From our CartPole experiments:
- **Basic Policy**: ~42 average reward (hardcoded, no learning)
- **REINFORCE**: ~180-200 average reward (learned, but sample inefficient)
- **DQN**: ~180-200 average reward (learned, more stable)

### Beyond This Notebook

**Modern RL Algorithms** (not covered in detail):
- **Actor-Critic**: Combines policy gradients with value functions
- **A3C/A2C**: Asynchronous/synchronous actor-critic methods
- **PPO**: Proximal Policy Optimization (used in ChatGPT, OpenAI Five)
- **SAC**: Soft Actor-Critic (high sample efficiency)
- **Rainbow DQN**: Combines all DQN improvements

**Advanced Topics**:
- Multi-agent RL
- Hierarchical RL
- Model-based RL
- Meta-RL
- Offline RL

### Real-World Applications

✅ Game playing (AlphaGo, Dota 2, StarCraft II)  
✅ Robotics (manipulation, locomotion)  
✅ Autonomous vehicles  
✅ Resource optimization (data centers, traffic control)  
✅ Recommender systems  
✅ Financial trading  
✅ Healthcare (treatment optimization)  

### Best Practices

1. **Start simple**: Test on simple environments first
2. **Monitor training**: Track rewards, losses, and exploration rate
3. **Tune hyperparameters**: Learning rate, batch size, network architecture matter
4. **Use proven implementations**: Libraries like Stable-Baselines3, RLlib, TF-Agents
5. **Be patient**: RL training can be unstable and time-consuming

### Next Steps

1. Try implementing DQN on different Gym environments (LunarLander, Breakout)
2. Experiment with different network architectures
3. Implement the DQN improvements (Double DQN, Dueling DQN)
4. Explore Actor-Critic methods (A2C, PPO)
5. Apply RL to a real-world problem!

In [ ]:
# Final cleanup
env.close()

print("=" * 70)
print("🎉 CONGRATULATIONS! 🎉")
print("=" * 70)
print("\nYou've successfully completed a comprehensive RL tutorial covering:")
print("  ✓ Policy Gradients (REINFORCE)")
print("  ✓ Q-Learning (Tabular)")
print("  ✓ Deep Q-Networks (DQN)")
print("\nYou've trained 3 different agents to solve CartPole!")
print("\n📚 Key Takeaways:")
print("  • RL agents learn through trial and error")
print("  • Policy-based methods optimize actions directly")
print("  • Value-based methods learn Q-values then derive policy")
print("  • Deep Learning enables RL for complex environments")
print("\n🚀 You're ready to explore more advanced RL algorithms!")
print("=" * 70)